### 说明
用于演示 AutoGen 框架下的多代理协作系统，具体展示了如何在旅行规划场景中实现两个具有不同角色的AI代理之间的协作。
1. **多代理系统架构**
    - 创建了两个具有明确角色分工的Agent：
        - `frontdesk_agent`（前台旅行代理）：专注于提供简洁、高效的旅行推荐
        - `concierge_agent`（酒店礼宾）：评估推荐质量并提供本地化改进建议
2. **业务场景实现**
    - 模拟了用户请求"我想计划一次巴黎之旅"的完整对话流程
    - 通过轮询式对话机制(`RoundRobinGroupChat`)实现代理间的有序交互
    - 设置了明确的终止条件(`TextMentionTermination`)：当礼宾回复"APPROVE"时结束对话
3. **框架特性展示**
    - 展示了AutoGen的流式响应处理(`run_stream`)
    - 演示了Azure AI服务集成配置
    - 体现了多代理协作的工作流程和消息传递机制

In [1]:
import os

from autogen_agentchat.agents import AssistantAgent
from autogen_core.models import UserMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient 
from autogen_ext.models.azure import AzureAIChatCompletionClient
from azure.core.credentials import AzureKeyCredential
from autogen_core import CancellationToken
from autogen_agentchat.base import TaskResult

from autogen_agentchat.messages import TextMessage
from autogen_agentchat.ui import Console


from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.teams import RoundRobinGroupChat

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
client = AzureAIChatCompletionClient(
    model="gpt-4.1-mini",
    endpoint="https://models.inference.ai.azure.com",
    # To authenticate with the model you will need to generate a personal access token (PAT) in your GitHub settings.
    # Create your PAT token by following instructions here: https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens
    credential=AzureKeyCredential(os.environ["GITHUB_TOKEN"]),
    model_info={
        "json_output": True,
        "structured_output": True, # 表明模型支持结构化输出。
        "function_calling": True,
        "vision": True,
        "family": "unknown",
    },
)

In [4]:
# 创建一个名为 frontdesk_agent 的 AssistantAgent 实例对象
frontdesk_agent = AssistantAgent(
    # 代理的唯一标识名称，通常用于系统内部调度或日志记录
    "planner_agent",
    # 指定该代理使用的模型客户端（如配置好的 OpenAI 或其他 LLM 客户端）
    model_client=client,
    # 代理的功能描述：这有助于编排者（Orchestrator）判断何时该调用这个特定的代理
    description="一个有帮助的助手，可以计划旅行。",
    # 系统消息（人设指令）：定义了 Agent 的角色、性格、行为准则和输出约束
    system_message="""
    你是一名有十年经验的前台旅行社代理，以简洁著称，因为你与许多顾客打交道。
我们的目标是为游客提供最好的活动和地点。
对每个回复只提供一个建议。
你全神贯注于手边的目标。
不要把时间浪费在闲聊上。
在完善一个想法的时候考虑别人的建议。""",
)


# 您是一位拥有十年经验的前台旅行社代理，在与众多客户打交道时以简洁著称。
# 目标是为旅行者提供最佳的活动和游览地点。
# 每个回复仅提供一条建议。
# 您全神贯注于当前的目标。
# 不要浪费时间闲聊。
# 在完善想法时考虑建议。


# 创建名为 concierge_agent 的助手代理实例
concierge_agent = AssistantAgent(
    # 代理的唯一名称标识，系统调度时会用到
    "concierge_agent",
    # 连接到大模型的客户端（如 OpenAI 或本地 LLM）
    model_client=client,
    # 代理的功能描述：告诉编排系统，这是一个能提供当地活动和地点建议的本地助手
    description="一个能建议当地活动或景点的本地助手。",
    # 系统消息（核心逻辑）：定义了代理的行为边界和评判标准
    system_message="""
    你是一位酒店礼宾员，对如何为旅客提供最本地化、最真实的体验有着自己的看法。
    目标是确定前台旅行社是否为旅行者推荐了最佳的非旅游体验。
    如果是，请回复“APPROVE”
    如果没有，请提供如何在不使用具体示例的情况下改进建议的见解。
    """,
)

# 您是一位酒店礼宾员，对如何为旅客提供最本地化、最真实的体验有着自己的看法。
# 目标是确定前台旅行社是否为旅行者推荐了最佳的非旅游体验。
# 如果是，请回复“批准”
# 如果没有，请提供如何在不使用具体示例的情况下改进建议的见解。


In [5]:
# --- 1. 定义终止条件 ---
# 实例化一个终止对象：当对话中出现 "APPROVE" 这个词时，系统会自动停止运行。
# 这通常用于由“审计者”代理发出的信号，表示建议已经达标。
termination = TextMentionTermination("APPROVE")
# --- 2. 组建团队与运行规则 ---
# 创建一个采用“轮询模式（Round Robin）”的群聊团队。
# [frontdesk_agent, concierge_agent]: 设定代理发言顺序，A 说完 B 说，以此往复。
# termination_condition: 将前面定义的终止条件绑定到团队中。
team = RoundRobinGroupChat(
    [frontdesk_agent, concierge_agent], termination_condition=termination
)
# --- 3. 启动异步流式任务 ---
# 使用异步迭代器运行团队任务。task 是交给团队的初始指令。
# run_stream 会实时返回代理之间的对话片段（Streaming），而不是等全部运行完才一次性返回。
async for message in team.run_stream(task="我想计划一次去巴黎的旅行。"): 
    # 检查当前消息是否为“任务最终结果对象”
    if isinstance(message, TaskResult):
        # 如果是结果对象，打印停止的原因（例如：是因为看到了 "APPROVE" 而停止的）
        print("Stop Reason:", message.stop_reason)
    else:
        # 如果是普通的代理对话内容，直接将其打印出来，方便我们实时观察 Agent 之间的交流
        print(message)

id='a994528c-2559-4d3a-981b-91dd41bce4f3' source='user' models_usage=None metadata={} created_at=datetime.datetime(2026, 6, 28, 16, 1, 58, 207990, tzinfo=datetime.timezone.utc) content='我想计划一次去巴黎的旅行。' type='TextMessage'
id='f508933e-0712-43ee-a0cf-099d0d288ecc' source='planner_agent' models_usage=RequestUsage(prompt_tokens=105, completion_tokens=29) metadata={} created_at=datetime.datetime(2026, 6, 28, 16, 2, 0, 889299, tzinfo=datetime.timezone.utc) content='建议您安排一次塞纳河游船，既能舒适欣赏巴黎地标，也能体验浪漫氛围。' type='TextMessage'
id='edac49b0-cc93-4a82-a49a-c6432b830892' source='concierge_agent' models_usage=RequestUsage(prompt_tokens=136, completion_tokens=102) metadata={} created_at=datetime.datetime(2026, 6, 28, 16, 2, 3, 873981, tzinfo=datetime.timezone.utc) content='这个建议虽然经典且舒适，但属于比较典型的旅游项目，缺少更加深入的本地化体验。为了提升推荐的质量，前台旅行社可以考虑引导旅客探索巴黎一些不那么知名、小众的社区文化、特色市集或本地艺术活动等，帮助旅客感受更加真实和多元的巴黎生活氛围。这样才能更好地满足追求独特、本地化体验的旅客需求。' type='TextMessage'
id='efff1e3b-364b-48ff-9ff0-660570735687' source='planner_agent' models_usag


---

**免责声明**：  
本文档使用AI翻译服务[Co-op Translator](https://github.com/Azure/co-op-translator)进行翻译。尽管我们努力确保翻译的准确性，但请注意，自动翻译可能包含错误或不准确之处。原始语言的文档应被视为权威来源。对于关键信息，建议使用专业人工翻译。我们不对因使用此翻译而产生的任何误解或误读承担责任。
